In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_MODEL = "gpt-4o-mini"

def get_llm(model: str = OPENAI_MODEL, temperature: float = 0.7) -> ChatOpenAI:
    """OpenAI API를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(api_key=OPENAI_API_KEY, model=model, temperature=temperature)

## 문제 2-1 : 콤마 구분 리스트 파서 활용

**출력 파서**: `CommaSeparatedListOutputParser`

### 문제
사용자가 관심있는 분야(예: "음식", "스포츠", "영화" 등)를 입력하면, 해당 분야와 관련된 **한국의 유명한 장소나 활동 5가지**를 콤마로 구분된 리스트로 출력하는 프로그램을 작성하세요.

### 요구사항
* `CommaSeparatedListOutputParser` 사용
* 사용자 입력을 받아 동적으로 처리
* 한국 관련 내용으로 추천

### 예시 입출력
* 입력: "음식"
* 출력:
`['명동 칼국수', '부산 돼지국밥', '전주 비빔밥', '춘천 닭갈비', '제주 흑돼지']`


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()
format_instructions = list_parser.get_format_instructions()

list_prompt = PromptTemplate(
    template="""당신은 한국 여행/문화 전문가입니다.
사용자가 관심 있는 분야와 관련된 **한국의 유명한 장소나 활동** 5가지를 추천하세요.

요구사항:
- 반드시 한국(대한민국)에 있는 장소나 한국에서 할 수 있는 활동만 추천하세요.
- 각 항목은 "지역명 + 장소/활동" 형태의 짧은 명사구로 작성하세요. (예: 전주 비빔밥)
- 설명, 번호, 따옴표 없이 정확히 5개만 한국어로 출력하세요.

관심 분야: {category}

{format_instructions}""",
    input_variables=["category"],
    partial_variables={"format_instructions": format_instructions},
)

list_chain = list_prompt | get_llm(temperature=0.7) | list_parser

def recommend_korea(category: str) -> list[str]:
    """관심 분야를 받아 한국의 유명한 장소/활동 5가지를 리스트로 반환합니다."""
    return list_chain.invoke({"category": category})

category = input("관심 분야를 입력하세요 (예: 음식, 스포츠, 영화): ").strip() or "음식"
result = recommend_korea(category)

print(f'입력: "{category}"')
print(f'출력: {result}')

## 문제 2-2 : 영화 리뷰 감정 분석기

**출력 파서**: `EnumOutputParser`

### 문제
영화 리뷰 텍스트를 입력받아 감정을 "긍정", "부정", "보통" 중 하나로 분류하는 시스템을 만드세요.

### 요구사항
* `EnumOutputParser`와 `Enum` 클래스 사용
* 여러 개의 테스트 리뷰로 검증
* 감정 분류 결과를 깔끔하게 출력

#### 테스트 리뷰 예시
1. "이 영화 정말 재미없어요. 시간 낭비였습니다."
2. "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!"
3. "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요."

> 분석실패했을 경우에 사용하는 `OutputFixingParser` 와 `EnumOutputParser` 를 같이 사용하는 것은 선택입니다.

In [ ]:
from enum import Enum
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import EnumOutputParser, OutputFixingParser


class Sentiment(str, Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"

enum_parser = EnumOutputParser(enum=Sentiment)
sentiment_llm = get_llm(temperature=0)
fixing_parser = OutputFixingParser.from_llm(parser=enum_parser, llm=sentiment_llm)

sentiment_prompt = PromptTemplate(
    template="""당신은 영화 리뷰 감정 분석 전문가입니다.
다음 영화 리뷰의 감정을 "긍정", "부정", "보통" 중 하나로 분류하세요.

분류 기준:
- 긍정: 영화를 칭찬하거나 만족감을 표현함
- 부정: 영화를 비판하거나 불만족을 표현함
- 보통: 특별히 좋지도 나쁘지도 않은 중립적인 평가

다른 설명 없이 분류 결과 단어 하나만 출력하세요.

리뷰: {review}

{format_instructions}""",
    input_variables=["review"],
    partial_variables={"format_instructions": enum_parser.get_format_instructions()},
)

sentiment_chain = sentiment_prompt | sentiment_llm | fixing_parser

test_reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
    "영상미는 압도적이었지만 결말이 너무 허무해서 아쉬웠어요.",
    "두 번 봐도 또 보고 싶은 인생 영화입니다.",
]

results = sentiment_chain.batch([{"review": r} for r in test_reviews])
for i, (review, sentiment) in enumerate(zip(test_reviews, results), start=1):
    print(f'{i}. 리뷰: "{review}"')
    print(f'   감정: {sentiment.value}({sentiment.name})\n')

summary = {s.value: results.count(s) for s in Sentiment}
print("요약:", ", ".join(f"{k} {v}개" for k, v in summary.items()))

## 문제 2-3 : 학생 정보 구조화 시스템

**출력 파서**: `PydanticOutputParser`

### 문제
학생의 자유 형식 자기소개를 입력받아 이름, 나이, 전공, 취미 리스트, 목표를 구조화된 형태로 추출하는 시스템을 만드세요.

### 요구사항
* `PydanticOutputParser`와 `BaseModel` 사용
* 각 필드에 적절한 타입과 설명 추가
* 자유 형식의 텍스트에서 정보 추출

### 예시 입력
"안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."

### 예시 출력
```python
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": ["게임하기", "영화보기", "코딩"],
    "goal": "훌륭한 개발자가 되는 것"
}
```


In [ ]:
import json
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class StudentInfo(BaseModel):
    name: str = Field(description="학생의 이름")
    age: int = Field(description="학생의 나이 (숫자만, 예: 22)")
    major: str = Field(description="학생의 전공 (예: 컴퓨터공학)")
    hobbies: list[str] = Field(description="학생의 취미 목록 (각 취미를 개별 항목으로)")
    goal: str = Field(description="학생의 앞으로의 목표 (예: 훌륭한 개발자가 되는 것)")

student_parser = PydanticOutputParser(pydantic_object=StudentInfo)

student_prompt = PromptTemplate(
    template="""다음 학생의 자기소개에서 정보를 추출하세요.
- 자기소개에 나온 표현을 최대한 그대로 사용하세요.
- 취미는 하나씩 나누어 리스트로 작성하세요.
- 목표는 "~하는 것" 형태의 명사구로 작성하세요.

자기소개: {introduction}

{format_instructions}""",
    input_variables=["introduction"],
    partial_variables={"format_instructions": student_parser.get_format_instructions()},
)

student_chain = student_prompt | get_llm(temperature=0) | student_parser

introduction = (
    "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. "
    "취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
)
student = student_chain.invoke({"introduction": introduction})

print(f'입력: "{introduction}"\n')
print(f'출력: ')
print(json.dumps(student.model_dump(), ensure_ascii=False, indent=4))

## 문제 2-4 : 여행 계획 분석기

**출력 파서**: `StructuredOutputParser`

### 문제
여행 후기나 계획 텍스트를 입력받아 여행지, 기간, 예산, 추천도(1-5점), 주요 활동 리스트를 구조화된 형태로 추출하는 시스템을 만드세요.

### 요구사항
* `StructuredOutputParser`와 `ResponseSchema` 사용
* 5개의 필드를 정의하여 정보 추출
* 자연어 텍스트에서 핵심 정보 파싱

### 예시 입력
"지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

### 예시 출력
```python
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": ["해운대 바다구경", "자갈치시장 회 먹기", "감천문화마을 구경"]
}
```

In [ ]:
import json
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

response_schemas = [
    ResponseSchema(name="destination", description="여행지 (예: 부산)"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="여행 예산 또는 총 경비 (예: 30만원)"),
    ResponseSchema(name="rating", description="여행 추천도. 1~5 사이의 정수를 문자열로 (예: \"4\")"),
    ResponseSchema(
        name="activities",
        description="주요 활동 리스트. 각 항목은 '장소 + 활동' 형태의 짧은 문구 (예: [\"해운대 바다구경\", \"자갈치시장 회 먹기\"])",
        type="List[string]",
    ),
]

travel_parser = StructuredOutputParser.from_response_schemas(response_schemas)

travel_prompt = PromptTemplate(
    template="""다음 여행 후기 또는 여행 계획 텍스트에서 핵심 정보를 추출하세요.
- 텍스트에 없는 정보는 "정보 없음"으로 작성하세요.
- 추천도는 텍스트의 만족도를 바탕으로 1~5 사이의 정수로 작성하세요.

텍스트: {text}

{format_instructions}""",
    input_variables=["text"],
    partial_variables={"format_instructions": travel_parser.get_format_instructions()},
)

travel_chain = travel_prompt | get_llm(temperature=0) | travel_parser

travel_text = (
    "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, "
    "자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. "
    "5점 만점에 4점 정도 줄 수 있을 것 같아요."
)
travel_info = travel_chain.invoke({"text": travel_text})

print(f'입력: "{travel_text}"\n')
print(f'출력: ')
print(json.dumps(travel_info, ensure_ascii=False, indent=4))